In [3]:
%pip install catboost


   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB 660.6 kB/s eta 0:02:32
   ---------------------------------------- 0.2/100.2 MB 2.4 MB/s eta 0:00:43
   ---------------------------------------- 1.0/100.2 MB 8.2 MB/s eta 0:00:13
   - -------------------------------------- 2.5/100.2 MB 14.7 MB/s eta 0:00:07
   - -------------------------------------- 3.2/100.2 MB 14.6 MB/s eta 0:00:07
   - -------------------------------------- 3.6/100.2 MB 16.3 MB/s eta 0:00:06
   -- ------------------------------------- 5.7/100.2 MB 19.3 MB/s eta 0:00:05
   -- ------------------------------------- 7.3/100.2 MB 20.4 MB/s eta 0:00:05
   --- ------------------------------------ 8.9/100.2 MB 22.0 MB/s eta 0:00:05
   --- ------------------------------------ 9.9/100.2 MB 21.8 MB/s eta 0:00:05
   ---- ----------------------------------- 11.6/100.2 MB 28.5 MB/s eta 0:00:04
   ----- ---------------------------------- 12.6/100.2 MB 27


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Si CatBoost no está instalado:


import re
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "Price_euros"
DROP_COLS = ["id", "laptop_ID", TARGET]


def add_features(df):
    df = df.copy()

    def number_before_unit(value, pattern):
        match = re.search(pattern, str(value), flags=re.IGNORECASE)
        return float(match.group(1)) if match else 0.0

    # Variables numéricas extraídas de columnas de texto
    df["Ram_GB"] = df["Ram"].apply(
        lambda x: number_before_unit(x, r"(\d+(?:\.\d+)?)\s*GB")
    )
    df["Weight_kg"] = df["Weight"].apply(
        lambda x: number_before_unit(x, r"(\d+(?:\.\d+)?)\s*kg")
    )
    df["CPU_GHz"] = df["Cpu"].apply(
        lambda x: number_before_unit(x, r"(\d+(?:\.\d+)?)\s*GHz")
    )

    # Resolución y características de pantalla
    resolutions = df["ScreenResolution"].astype(str).str.extract(
        r"(\d{3,4})x(\d{3,4})"
    )
    df["Screen_width"] = pd.to_numeric(resolutions[0], errors="coerce").fillna(0)
    df["Screen_height"] = pd.to_numeric(resolutions[1], errors="coerce").fillna(0)
    df["Screen_pixels"] = df["Screen_width"] * df["Screen_height"]
    df["Touchscreen"] = df["ScreenResolution"].str.contains(
        "touchscreen", case=False, na=False
    ).astype(int)
    df["IPS"] = df["ScreenResolution"].str.contains(
        "IPS", case=False, na=False
    ).astype(int)

    # Capacidad total de almacenamiento, normalizada a GB
    def storage_gb(value):
        matches = re.findall(
            r"(\d+(?:\.\d+)?)\s*(TB|GB)", str(value), flags=re.IGNORECASE
        )
        return sum(
            float(amount) * (1024 if unit.upper() == "TB" else 1)
            for amount, unit in matches
        )

    df["Storage_GB"] = df["Memory"].apply(storage_gb)
    df["Has_SSD"] = df["Memory"].str.contains("SSD", case=False, na=False).astype(int)
    df["Has_HDD"] = df["Memory"].str.contains("HDD", case=False, na=False).astype(int)
    df["Has_Flash"] = df["Memory"].str.contains(
        "Flash", case=False, na=False
    ).astype(int)

    # Marcas como variables adicionales; se conservan también los textos originales
    df["CPU_brand"] = df["Cpu"].str.extract(
        r"^(Intel|AMD|Samsung)", flags=re.IGNORECASE
    )[0].fillna("Other")
    df["GPU_brand"] = df["Gpu"].str.extract(
        r"^(Intel|AMD|Nvidia|ARM)", flags=re.IGNORECASE
    )[0].fillna("Other")

    # CatBoost requiere categorías como texto y sin valores nulos
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].fillna("Unknown").astype(str)

    return df


X = add_features(train.drop(columns=[TARGET]))
y = train[TARGET]
X_test = add_features(test)

# No usar id ni laptop_ID: son identificadores, no especificaciones del equipo.
X = X.drop(columns=["id", "laptop_ID"], errors="ignore")
X_test = X_test.drop(columns=["id", "laptop_ID"], errors="ignore")

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

# Validación para estimar el MAE
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42
)

validation_model = CatBoostRegressor(
    loss_function="MAE",
    iterations=3000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

validation_model.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=200,
    verbose=200,
)

valid_predictions = validation_model.predict(X_valid)
print("MAE de validación:", mean_absolute_error(y_valid, valid_predictions))

# Entrenar con todos los datos y crear el archivo para Kaggle
best_iterations = validation_model.get_best_iteration()
final_iterations = max(best_iterations, 1) if best_iterations is not None else 1500

model = CatBoostRegressor(
    loss_function="MAE",
    iterations=final_iterations,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

model.fit(train_features := X, y, cat_features=categorical_features)

submission = pd.DataFrame({
    "id": test["id"],
    "Price_euros": model.predict(X_test),
})
submission.to_csv("submission.csv", index=False)

print(submission.head())
print(f"Archivo guardado: submission.csv ({len(submission)} filas)")

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'